# Dual-Expert Attention Model Training Pipeline
## APTOS2019 Blindness Detection Dataset

Training notebook for the Dual-Expert Attention model (SE + CBAM experts with learnable fusion).  
All experimental conditions are identical to the CViTS-Net experiment for fair comparison.

## 1. Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.getcwd())

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import auc
import time
import json

# Reuse the same dataset loader and metrics as CViTS-Net experiment
from dataset_loader import get_data_loaders
from metrics import MetricsCalculator

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1024**3:.1f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Set seeds (identical to CViTS experiment)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("\n✓ Environment configured successfully")

## 2. Configuration

In [ ]:
# ============================================================
# All hyperparameters identical to CViTS-Net experiment
# ============================================================
CONFIG = {
    'experiment_name': 'dual_expert',
    'dataset_path': 'APTOS2019',
    'num_classes': 5,
    'image_size': 224,
    'batch_size': 16,
    'epochs': 100,
    'learning_rate': 0.001,
    'weight_decay': 0.0001,
    'optimizer': 'AdamW',
    'loss_function': 'CrossEntropyLoss',
    'scheduler': None,          # No scheduler (same as CViTS)
    'augmentation': None,       # No augmentation (same as CViTS)
    'preprocessing': None,      # No preprocessing (same as CViTS)
    'checkpoint_monitor': 'val_qwk',
    'checkpoint_mode': 'max',
}

# Output directories
output_dir = Path('results/dual_expert')
model_dir = Path('trained_model/dual_expert')
plots_dir = output_dir / 'plots'
logs_dir = output_dir / 'logs'

output_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\n✓ Output directories created")

## 3. Dataset Loading

Using the **exact same** dataset loader, splits (random_state=42), and batch size as the CViTS experiment.

In [ ]:
print("\n" + "="*80)
print("Loading APTOS2019 Dataset (identical to CViTS experiment)")
print("="*80)

max_retries = 3
for attempt in range(max_retries):
    try:
        print(f"\nAttempt {attempt + 1}/{max_retries}...")
        
        train_dataset, val_dataset, test_dataset, class_weights, \
        (X_train, y_train, X_val, y_val, X_test, y_test) = \
        get_data_loaders(dataset_path=CONFIG['dataset_path'], batch_size=CONFIG['batch_size'])
        
        print("\u2713 Dataset loaded successfully!")
        print(f"  Train samples: {len(X_train)}")
        print(f"  Val samples: {len(X_val)}")
        print(f"  Test samples: {len(X_test)}")
        break
        
    except Exception as e:
        print(f"Error: {str(e)}")
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt
            print(f"Retrying in {wait_time} seconds...")
            time.sleep(wait_time)
        else:
            raise

## 4. Data Augmentation

**No augmentation** is applied, matching the CViTS experiment exactly.  
Images are loaded as 224\u00d7224 uint8, converted to float32 tensors by the DataLoader.

In [ ]:
# ============================================================
# Data Augmentation: NONE
# ============================================================
# The CViTS experiment does not use any data augmentation.
# To ensure a fair comparison, no augmentation is applied here either.
# Images are only resized to 224x224 during loading (PIL.resize).

print("Data Augmentation: None (matching CViTS experiment)")
print("Preprocessing: None (matching CViTS experiment)")
print("Image format: 224x224x3 uint8 -> float32 tensor [0-255]")

## 5. Model Definition \u2014 Dual-Expert Attention

Architecture:
1. **Backbone CNN** \u2014 Feature extraction (Conv blocks with BatchNorm)
2. **Expert 1** \u2014 Squeeze-and-Excitation (SE) Attention
3. **Expert 2** \u2014 Convolutional Block Attention Module (CBAM)
4. **Learnable Weighted Fusion** \u2014 `fused = \u03b1 \u00d7 SE + \u03b2 \u00d7 CBAM`
5. **Classification Head** \u2014 GAP \u2192 FC \u2192 5 classes

In [ ]:
# ============================================================
# Squeeze-and-Excitation (SE) Block
# ============================================================
class SEBlock(nn.Module):
    """Squeeze-and-Excitation attention block."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


# ============================================================
# CBAM: Channel Attention + Spatial Attention
# ============================================================
class ChannelAttention(nn.Module):
    """Channel attention sub-module for CBAM."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)


class SpatialAttention(nn.Module):
    """Spatial attention sub-module for CBAM."""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))


class CBAMBlock(nn.Module):
    """Convolutional Block Attention Module (CBAM)."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.channel_att = ChannelAttention(channels, reduction)
        self.spatial_att = SpatialAttention()

    def forward(self, x):
        x = x * self.channel_att(x)
        x = x * self.spatial_att(x)
        return x


# ============================================================
# Backbone CNN
# ============================================================
class ConvBlock(nn.Module):
    """Conv -> BN -> ReLU -> Conv -> BN -> ReLU block."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut for residual connection
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = F.relu(out + self.shortcut(x))
        return out


class BackboneCNN(nn.Module):
    """CNN backbone for feature extraction: 224x224x3 -> 7x7x512."""
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),  # 112x112
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),                  # 56x56
        )
        self.layer1 = nn.Sequential(ConvBlock(64, 64), ConvBlock(64, 64))
        self.layer2 = nn.Sequential(ConvBlock(64, 128, stride=2), ConvBlock(128, 128))
        self.layer3 = nn.Sequential(ConvBlock(128, 256, stride=2), ConvBlock(256, 256))
        self.layer4 = nn.Sequential(ConvBlock(256, 512, stride=2), ConvBlock(512, 512))

    def forward(self, x):
        x = self.stem(x)    # (B, 64, 56, 56)
        x = self.layer1(x)  # (B, 64, 56, 56)
        x = self.layer2(x)  # (B, 128, 28, 28)
        x = self.layer3(x)  # (B, 256, 14, 14)
        x = self.layer4(x)  # (B, 512, 7, 7)
        return x


# ============================================================
# Dual-Expert Attention Model
# ============================================================
class DualExpertAttentionModel(nn.Module):
    """
    Dual-Expert Attention Model.
    
    Architecture:
        1. Backbone CNN extracts feature maps
        2. Expert 1: SE Attention
        3. Expert 2: CBAM Attention
        4. Learnable weighted fusion: fused = alpha * SE + beta * CBAM
        5. GAP -> FC -> 5 classes
    """
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = BackboneCNN()
        
        # Two parallel attention experts
        self.se_expert = SEBlock(512, reduction=16)
        self.cbam_expert = CBAMBlock(512, reduction=16)
        
        # Learnable fusion weights (initialized to 0.5 each)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.beta = nn.Parameter(torch.tensor(0.5))
        
        # Classification head
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.5)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        # Normalize to [0,1] — same logic as CViTS-Net
        if x.dtype == torch.uint8:
            x = x.float() / 255.0
        
        # Backbone feature extraction
        features = self.backbone(x)           # (B, 512, 7, 7)
        
        # Expert 1: SE attention
        se_features = self.se_expert(features)    # (B, 512, 7, 7)
        
        # Expert 2: CBAM attention
        cbam_features = self.cbam_expert(features)  # (B, 512, 7, 7)
        
        # Learnable weighted fusion
        fused = self.alpha * se_features + self.beta * cbam_features  # (B, 512, 7, 7)
        
        # Classification head
        pooled = self.gap(fused).flatten(1)   # (B, 512)
        pooled = self.dropout(pooled)
        output = self.classifier(pooled)      # (B, num_classes)
        return output


def count_parameters(model):
    """Count total trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Build model
print("\n" + "="*80)
print("Building Dual-Expert Attention Model")
print("="*80)

model = DualExpertAttentionModel(num_classes=CONFIG['num_classes'])
model = model.to(device)

# Same optimizer as CViTS experiment
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

total_params = count_parameters(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")
print(f"Device: {device}")
print(f"\nModel Architecture:")
print(f"  Backbone: Custom CNN (Conv blocks with residual connections)")
print(f"  Expert 1: Squeeze-and-Excitation (SE)")
print(f"  Expert 2: CBAM (Channel + Spatial Attention)")
print(f"  Fusion: Learnable weighted (alpha={model.alpha.item():.2f}, beta={model.beta.item():.2f})")
print(f"  Head: GAP -> Dropout(0.5) -> FC(512, 5)")
print(f"\n✓ Model built successfully!")

## 6. Training Loop

In [ ]:
print("\n" + "="*80)
print(f"Starting Training ({CONFIG['epochs']} epochs)")
print("="*80)

# Initialize metrics history (identical structure to CViTS)
history = {
    'loss': {'train': [], 'val': []},
    'accuracy': {'train': [], 'val': []},
    'precision': {'train': [], 'val': []},
    'recall': {'train': [], 'val': []},
    'f1_score': {'train': [], 'val': []},
    'specificity': {'train': [], 'val': []},
    'roc_auc': {'train': [], 'val': []},
    'qwk': {'train': [], 'val': []}
}

epochs = CONFIG['epochs']

# Best model tracking (monitor val_qwk, mode=max) — identical to CViTS
best_val_qwk = -1.0
best_epoch = 0

# Training loop
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    print("-" * 80)
    
    # ---- Training phase ----
    model.train()
    train_metrics_calc = MetricsCalculator(num_classes=CONFIG['num_classes'])
    train_loss = 0.0
    num_batches = 0
    
    for batch_images, batch_labels in train_dataset:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        predictions = model(batch_images)
        loss = F.cross_entropy(predictions, batch_labels)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pred_proba = F.softmax(predictions, dim=1).detach().cpu().numpy()
        train_metrics_calc.update(batch_labels.cpu().numpy(), pred_proba)
        num_batches += 1
    
    train_loss = train_loss / num_batches
    train_metrics = train_metrics_calc.compute_metrics()
    
    # ---- Validation phase ----
    model.eval()
    val_metrics_calc = MetricsCalculator(num_classes=CONFIG['num_classes'])
    val_loss = 0.0
    num_val_batches = 0
    
    with torch.no_grad():
        for batch_images, batch_labels in val_dataset:
            batch_images = batch_images.to(device)
            batch_labels = batch_labels.to(device)
            
            predictions = model(batch_images)
            loss = F.cross_entropy(predictions, batch_labels)
            
            val_loss += loss.item()
            pred_proba = F.softmax(predictions, dim=1).cpu().numpy()
            val_metrics_calc.update(batch_labels.cpu().numpy(), pred_proba)
            num_val_batches += 1
    
    val_loss = val_loss / num_val_batches
    val_metrics = val_metrics_calc.compute_metrics()
    
    # Store history
    history['loss']['train'].append(float(train_loss))
    history['loss']['val'].append(float(val_loss))
    for metric_name in ['accuracy', 'precision', 'recall', 'f1_score', 'specificity', 'roc_auc', 'qwk']:
        if metric_name in train_metrics:
            history[metric_name]['train'].append(train_metrics[metric_name])
            history[metric_name]['val'].append(val_metrics[metric_name])
    
    # Current val QWK
    current_val_qwk = val_metrics.get('qwk', 0)
    
    # Print epoch results
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    print(f"Train Acc: {train_metrics.get('accuracy', 0):.4f} | Val Acc: {val_metrics.get('accuracy', 0):.4f}")
    print(f"Train F1: {train_metrics.get('f1_score', 0):.4f} | Val F1: {val_metrics.get('f1_score', 0):.4f}")
    print(f"Train QWK: {train_metrics.get('qwk', 0):.4f} | Val QWK: {current_val_qwk:.4f}")
    
    # Save best model based on val_qwk (monitor="val_qwk", mode="max") — identical to CViTS
    if current_val_qwk > best_val_qwk:
        best_val_qwk = current_val_qwk
        best_epoch = epoch + 1
        best_model_path = model_dir / 'best_model_dual_expert.pth'
        torch.save({
            'epoch': best_epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_qwk': best_val_qwk,
            'val_loss': val_loss,
        }, str(best_model_path))
        print(f"✓ Best model saved (val_QWK: {best_val_qwk:.4f}, epoch: {best_epoch})")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = model_dir / f'checkpoint_epoch_{epoch + 1:03d}.pth'
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, str(checkpoint_path))
        print(f"Checkpoint saved: {checkpoint_path}")

# Print fusion weights after training
print(f"\nLearned fusion weights: alpha={model.alpha.item():.4f}, beta={model.beta.item():.4f}")
print("\n" + "="*80)
print("Training completed!")
print(f"Best model: Epoch {best_epoch} (val_QWK: {best_val_qwk:.4f})")
print("="*80)

## 7. Validation Metrics

All 7 metrics are computed during training (see history above):  
Accuracy, Precision, Recall, F1-score, Specificity, ROC-AUC, QWK

In [ ]:
print("\n" + "="*80)
print("Validation Metrics Summary (Last Epoch)")
print("="*80)

print(f"\n  Val Loss:        {history['loss']['val'][-1]:.4f}")
print(f"  Val Accuracy:    {history['accuracy']['val'][-1]:.4f}")
print(f"  Val Precision:   {history['precision']['val'][-1]:.4f}")
print(f"  Val Recall:      {history['recall']['val'][-1]:.4f}")
print(f"  Val F1 Score:    {history['f1_score']['val'][-1]:.4f}")
print(f"  Val Specificity: {history['specificity']['val'][-1]:.4f}")
print(f"  Val ROC-AUC:     {history['roc_auc']['val'][-1]:.4f}")
print(f"  Val QWK:         {history['qwk']['val'][-1]:.4f}")

# Best epoch analysis
best_epoch_qwk = int(np.argmax(history['qwk']['val'])) + 1
best_epoch_loss = int(np.argmin(history['loss']['val'])) + 1
best_epoch_acc = int(np.argmax(history['accuracy']['val'])) + 1

print(f"\n  Best Val QWK:    {max(history['qwk']['val']):.4f} (Epoch {best_epoch_qwk})")
print(f"  Best Val Loss:   {min(history['loss']['val']):.4f} (Epoch {best_epoch_loss})")
print(f"  Best Val Acc:    {max(history['accuracy']['val']):.4f} (Epoch {best_epoch_acc})")

## 8. Checkpoint Saving

In [ ]:
print("\n" + "="*80)
print("Saving Final Model and Training History")
print("="*80)

# Save final model
final_model_path = model_dir / 'dual_expert_final.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'alpha': model.alpha.item(),
    'beta': model.beta.item(),
}, str(final_model_path))
print(f"\u2713 Final model saved: {final_model_path}")

# Save training history
def convert_types(obj):
    if isinstance(obj, dict):
        return {k: convert_types(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_types(item) for item in obj]
    elif isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    else:
        return obj

history_path = logs_dir / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump(convert_types(history), f, indent=4)
print(f"\u2713 Training history saved: {history_path}")

# Save config
config_path = logs_dir / 'config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=4)
print(f"\u2713 Config saved: {config_path}")

## 9. Test Evaluation

Load the **best saved model** (highest val_QWK) and evaluate once on the test set.

In [ ]:
print("\n" + "="*80)
print("Evaluating on Test Set (using best model by val_QWK)")
print("="*80)

# Load best model (saved based on highest val_QWK)
best_model_path = model_dir / 'best_model_dual_expert.pth'
if best_model_path.exists():
    best_checkpoint = torch.load(str(best_model_path), map_location=device)
    model.load_state_dict(best_checkpoint['model_state_dict'])
    print(f"✓ Loaded best model from Epoch {best_checkpoint['epoch']} (val_QWK: {best_checkpoint['val_qwk']:.4f})")
else:
    print("⚠ Best model not found, using current model weights")
    best_checkpoint = {'epoch': epochs, 'val_qwk': history['qwk']['val'][-1], 'val_loss': history['loss']['val'][-1]}

model.eval()
test_metrics_calc = MetricsCalculator(num_classes=CONFIG['num_classes'])
test_loss = 0.0
num_test_batches = 0

with torch.no_grad():
    for batch_images, batch_labels in test_dataset:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)
        
        predictions = model(batch_images)
        loss = F.cross_entropy(predictions, batch_labels)
        
        test_loss += loss.item()
        pred_proba = F.softmax(predictions, dim=1).cpu().numpy()
        test_metrics_calc.update(batch_labels.cpu().numpy(), pred_proba)
        num_test_batches += 1

test_loss = test_loss / num_test_batches
test_metrics = test_metrics_calc.compute_metrics()

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_metrics.get('accuracy', 0):.4f}")
print(f"Test Precision: {test_metrics.get('precision', 0):.4f}")
print(f"Test Recall: {test_metrics.get('recall', 0):.4f}")
print(f"Test F1 Score: {test_metrics.get('f1_score', 0):.4f}")
print(f"Test Specificity: {test_metrics.get('specificity', 0):.4f}")
print(f"Test ROC-AUC: {test_metrics.get('roc_auc', 0):.4f}")
print(f"Test QWK: {test_metrics.get('qwk', 0):.4f}")

# Get confusion matrix and ROC curves
cm = test_metrics_calc.get_confusion_matrix()
roc_data = test_metrics_calc.get_roc_curves()

# Store test results
test_history = {
    'loss': float(test_loss),
    'metrics': test_metrics,
    'confusion_matrix': cm.tolist(),
    'best_model_epoch': best_checkpoint['epoch'],
    'best_model_val_qwk': float(best_checkpoint['val_qwk'])
}

# Save test results
test_results = {
    'training_history': convert_types(history),
    'test_results': convert_types(test_history)
}
results_path = logs_dir / 'full_results.json'
with open(results_path, 'w') as f:
    json.dump(test_results, f, indent=4)
print(f"\n✓ Full results saved: {results_path}")

# Save best model metrics JSON
best_model_metrics = {
    'best_model_epoch': int(best_checkpoint['epoch']),
    'best_model_val_qwk': float(best_checkpoint['val_qwk']),
    'test_loss': float(test_loss),
    'test_metrics': {
        'accuracy': float(test_metrics.get('accuracy', 0)),
        'precision': float(test_metrics.get('precision', 0)),
        'recall': float(test_metrics.get('recall', 0)),
        'f1_score': float(test_metrics.get('f1_score', 0)),
        'specificity': float(test_metrics.get('specificity', 0)),
        'roc_auc': float(test_metrics.get('roc_auc', 0)),
        'qwk': float(test_metrics.get('qwk', 0))
    }
}
metrics_path = logs_dir / 'best_model_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(best_model_metrics, f, indent=4)
print(f"✓ Best model metrics saved: {metrics_path}")

print(f"\n✓ Test evaluation complete (best model from epoch {best_checkpoint['epoch']})")

## 10. Visualization

In [ ]:
print("\n" + "="*80)
print("Generating Visualizations")
print("="*80)

epochs_range = range(1, len(history['loss']['train']) + 1)
class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# ---- 1. Training vs Validation Loss ----
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs_range, history['loss']['train'], 'b-', linewidth=2, label='Train Loss')
ax.plot(epochs_range, history['loss']['val'], 'r-', linewidth=2, label='Validation Loss')
best_loss_idx = int(np.argmin(history['loss']['val']))
ax.axvline(x=best_loss_idx + 1, color='green', linestyle='--', alpha=0.7,
           label=f'Min Val Loss: {history["loss"]["val"][best_loss_idx]:.4f} (Epoch {best_loss_idx + 1})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Dual-Expert — Training vs Validation Loss', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(plots_dir / 'loss_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: loss_curve.png")

# ---- 2. Training vs Validation Accuracy ----
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs_range, history['accuracy']['train'], 'b-', linewidth=2, label='Train Accuracy')
ax.plot(epochs_range, history['accuracy']['val'], 'r-', linewidth=2, label='Validation Accuracy')
best_acc_idx = int(np.argmax(history['accuracy']['val']))
ax.scatter([best_acc_idx + 1], [history['accuracy']['val'][best_acc_idx]], color='green', s=100, zorder=5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Dual-Expert — Training vs Validation Accuracy', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(plots_dir / 'accuracy_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: accuracy_curve.png")

# ---- 3. F1-score Curve ----
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs_range, history['f1_score']['train'], 'b-', linewidth=2, label='Train F1')
ax.plot(epochs_range, history['f1_score']['val'], 'r-', linewidth=2, label='Validation F1')
best_f1_idx = int(np.argmax(history['f1_score']['val']))
best_f1_val = history['f1_score']['val'][best_f1_idx]
ax.axvline(x=best_f1_idx + 1, color='green', linestyle='--', alpha=0.7,
           label=f'Best Val F1: {best_f1_val:.4f} (Epoch {best_f1_idx + 1})')
ax.scatter([best_f1_idx + 1], [best_f1_val], color='green', s=100, zorder=5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('F1-score', fontsize=12)
ax.set_title('Dual-Expert — F1-score over Training Epochs', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(plots_dir / 'f1_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: f1_curve.png")

# ---- 4. QWK Curve ----
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(epochs_range, history['qwk']['train'], 'b-', linewidth=2, label='Train QWK')
ax.plot(epochs_range, history['qwk']['val'], 'r-', linewidth=2, label='Validation QWK')
best_qwk_idx = int(np.argmax(history['qwk']['val']))
best_qwk_val = history['qwk']['val'][best_qwk_idx]
ax.axvline(x=best_qwk_idx + 1, color='green', linestyle='--', alpha=0.7,
           label=f'Best Val QWK: {best_qwk_val:.4f} (Epoch {best_qwk_idx + 1})')
ax.scatter([best_qwk_idx + 1], [best_qwk_val], color='green', s=100, zorder=5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Quadratic Weighted Kappa (QWK)', fontsize=12)
ax.set_title('Dual-Expert — QWK over Training Epochs', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(plots_dir / 'qwk_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: qwk_curve.png")

# ---- 5. Confusion Matrix ----
fig, ax = plt.subplots(figsize=(10, 8))
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=cm, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Normalized Count'}, ax=ax)
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_title('Dual-Expert — Confusion Matrix (Best Model)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(plots_dir / 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: confusion_matrix.png")

# ---- 6. ROC Curve ----
if roc_data:
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.Set1(np.linspace(0, 1, len(roc_data)))
    
    for i, (class_name, roc_vals) in enumerate(roc_data.items()):
        fpr = roc_vals['fpr']
        tpr = roc_vals['tpr']
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=colors[i], lw=2,
                label=f'{class_names[i]} (AUC = {roc_auc_val:.3f})')
    
    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Classifier')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('Dual-Expert — ROC Curves', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(plots_dir / 'roc_curve.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: roc_curve.png")

print("\n✓ All visualizations generated successfully!")

## Summary

In [ ]:
print("\n" + "="*80)
print("DUAL-EXPERT EXPERIMENT COMPLETED!")
print("="*80)

print(f"\n📊 Results Summary:")
print(f"\n  Model: Dual-Expert Attention (SE + CBAM, learnable fusion)")
print(f"  Parameters: {total_params:,}")
print(f"  Learned fusion: alpha={model.alpha.item():.4f}, beta={model.beta.item():.4f}")

print(f"\n  Final Training Metrics (Epoch {epochs}):")
print(f"    Loss: {history['loss']['train'][-1]:.4f}")
print(f"    Accuracy: {history['accuracy']['train'][-1]:.4f}")
print(f"    F1 Score: {history['f1_score']['train'][-1]:.4f}")
print(f"    QWK: {history['qwk']['train'][-1]:.4f}")

print(f"\n  Final Validation Metrics (Epoch {epochs}):")
print(f"    Loss: {history['loss']['val'][-1]:.4f}")
print(f"    Accuracy: {history['accuracy']['val'][-1]:.4f}")
print(f"    F1 Score: {history['f1_score']['val'][-1]:.4f}")
print(f"    QWK: {history['qwk']['val'][-1]:.4f}")

print(f"\n  Best Model: Epoch {best_model_metrics['best_model_epoch']} (val_QWK: {best_model_metrics['best_model_val_qwk']:.4f})")
print(f"\n  Test Set Metrics (Best Model):")
print(f"    Loss: {test_loss:.4f}")
print(f"    Accuracy: {test_metrics['accuracy']:.4f}")
print(f"    Precision: {test_metrics['precision']:.4f}")
print(f"    Recall: {test_metrics['recall']:.4f}")
print(f"    F1 Score: {test_metrics['f1_score']:.4f}")
print(f"    Specificity: {test_metrics['specificity']:.4f}")
print(f"    ROC-AUC: {test_metrics['roc_auc']:.4f}")
print(f"    QWK: {test_metrics['qwk']:.4f}")

print(f"\n📁 Output Files:")
print(f"  Best Model: {model_dir / 'best_model_dual_expert.pth'}")
print(f"  Final Model: {model_dir / 'dual_expert_final.pth'}")
print(f"  Plots: {plots_dir}/")
print(f"  Results: {logs_dir / 'full_results.json'}")
print(f"  Best Metrics: {logs_dir / 'best_model_metrics.json'}")

print(f"\n✅ Experiment completed successfully!")
print("=" * 80)